<a href="https://colab.research.google.com/github/vineetkrtyagi/CudaProgramming/blob/main/Matrix_multiplication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Tue Jun 16 10:21:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [3]:
%%writefile main.cpp
#include <iostream>
using namespace std;

int main() {
    cout << "Hello, World!" << endl;
    return 0;
}

Writing main.cpp


In [4]:
!g++ main.cpp -o main
!./main

Hello, World!


In [15]:
%%writefile matrix_mul.cu

#include <cuda_runtime.h>
#include <iostream>
#include <vector>
#include <chrono>
#include <cmath>

#define CUDA_CHECK(call)                              \
do {                                                  \
    cudaError_t err = call;                           \
    if (err != cudaSuccess) {                         \
        std::cerr << "CUDA Error: "                   \
                  << cudaGetErrorString(err)          \
                  << std::endl;                       \
        exit(EXIT_FAILURE);                           \
    }                                                 \
} while (0)


// GPU kernel
__global__ void matrixMulKernel(
        const float* A,
        const float* B,
        float* C,
        int N)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if(row < N && col < N)
    {
        float sum = 0.0f;

        for(int k=0;k<N;k++)
        {
            sum += A[row * N + k] * B[k * N + col];
        }

        C[row * N + col] = sum;
    }
}


int main()
{
    int N = 512;
    size_t bytes = N * N * sizeof(float);

    std::cout<<"Matrix size : "<<N<<" x "<<N<<std::endl;

    // Host matrices
    std::vector<float> h_A(N*N);
    std::vector<float> h_B(N*N);
    std::vector<float> h_C_gpu(N*N);
    std::vector<float> h_C_cpu(N*N);

    // Initialize
    for(int i=0;i<N*N;i++)
    {
        h_A[i]=1.0f;
        h_B[i]=2.0f;
    }

    // Device pointers
    float *d_A,*d_B,*d_C;

    CUDA_CHECK(cudaMalloc((void**)&d_A,bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_B,bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_C,bytes));

    CUDA_CHECK(cudaMemcpy(d_A,h_A.data(),bytes,cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B,h_B.data(),bytes,cudaMemcpyHostToDevice));

    dim3 threadsPerBlock(16,16);
    dim3 blocksPerGrid(
        (N + threadsPerBlock.x -1)/threadsPerBlock.x,
        (N + threadsPerBlock.y -1)/threadsPerBlock.y);

    // GPU timing
    cudaEvent_t start,stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    matrixMulKernel<<<blocksPerGrid,threadsPerBlock>>>(
        d_A,d_B,d_C,N);

    CUDA_CHECK(cudaGetLastError());

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTime=0;
    cudaEventElapsedTime(&gpuTime,start,stop);

    CUDA_CHECK(cudaMemcpy(
        h_C_gpu.data(),
        d_C,
        bytes,
        cudaMemcpyDeviceToHost));

    // CPU timing
    auto cpu_start = std::chrono::high_resolution_clock::now();

    for(int row=0;row<N;row++)
    {
        for(int col=0;col<N;col++)
        {
            float sum=0;

            for(int k=0;k<N;k++)
            {
                sum += h_A[row*N+k]*h_B[k*N+col];
            }

            h_C_cpu[row*N+col]=sum;
        }
    }

    auto cpu_end = std::chrono::high_resolution_clock::now();

    std::chrono::duration<double,std::milli> cpuTime =
            cpu_end-cpu_start;

    // Verify
    bool passed=true;

    for(int i=0;i<N*N;i++)
    {
        if(fabs(h_C_gpu[i]-h_C_cpu[i])>1e-5)
        {
            passed=false;
            break;
        }
    }

    std::cout<<"GPU Time : "<<gpuTime<<" ms"<<std::endl;
    std::cout<<"CPU Time : "<<cpuTime.count()<<" ms"<<std::endl;

    if(passed)
        std::cout<<"Verification PASSED\n";
    else
        std::cout<<"Verification FAILED\n";

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}

Writing matrix_mul.cu


In [16]:
!nvcc -o matrix_mul matrix_mul.cu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [19]:
!./matrix_mul

Matrix size : 512 x 512
GPU Time : 1.2601 ms
CPU Time : 1105.21 ms
Verification PASSED
